# Setup and Loading Data

In [2]:
import os
import sys
import json
import numpy as np
from collections import defaultdict

sys.path.append(os.path.abspath('..'))
from src.preprocessing import IRPreprocessor
from src.bm25_ranker import BM25Engine
from src.metrics import calculate_average_precision, calculate_map

RAW_DIR = '../data/raw/'

with open(os.path.join(RAW_DIR, 'cran_docs.json'), 'r') as f:
    docs = json.load(f)
with open(os.path.join(RAW_DIR, 'cran_queries.json'), 'r') as f:
    queries = json.load(f)
with open(os.path.join(RAW_DIR, 'cran_qrels.json'), 'r') as f:
    qrels = json.load(f)

print("Data loaded for BM25 Baseline.")

Data loaded for BM25 Baseline.


# Mapping Ground Truth

In [3]:
ground_truth = defaultdict(set)
for qrel in qrels:
    query_id = str(int(qrel['query_num'])) # Normalize IDs
    doc_id = str(int(qrel['id']))
    # adjustable relevance threshold
    ground_truth[query_id].add(doc_id)

print(f"Mapped ground truth for {len(ground_truth)} queries.")

Mapped ground truth for 225 queries.


# Tokenization

In [4]:
preprocessor = IRPreprocessor()

# Tokenize corpus and queries
tokenized_corpus = [preprocessor.tokenize_for_bm25(doc['body']) for doc in docs]
doc_ids = [str(int(doc['id'])) for doc in docs]

print("Corpus tokenized for Okapi BM25.")

Corpus tokenized for Okapi BM25.


# Evaluation

In [5]:
bm25_engine = BM25Engine(tokenized_corpus)

ap_scores = []

# Evaluate each query
for query in queries:
    raw_id = query.get('query_number') or query.get('query number') or query.get('id') or query.get('query_num')
    if raw_id is None:
        continue
        
    q_id = str(int(raw_id))
    q_text = str(query.get('query', ''))
    
    if q_id not in ground_truth:
        continue # Skip if no relevance judgments for this query
        
    tokenized_query = preprocessor.tokenize_for_bm25(q_text)
    
    # Get scores for all documents
    scores = bm25_engine.get_scores(tokenized_query)
    
    # Sort document IDs by score descending
    ranked_indices = np.argsort(scores)[::-1]
    retrieved_doc_ids = [doc_ids[i] for i in ranked_indices]
    
    # Calculate Average Precision
    relevant_docs = ground_truth[q_id]
    ap = calculate_average_precision(retrieved_doc_ids, relevant_docs)
    ap_scores.append(ap)

# Calculate Mean Average Precision (MAP)
baseline_map = calculate_map(ap_scores)
print(f"Baseline BM25 Mean Average Precision (MAP): {baseline_map:.4f}")

Baseline BM25 Mean Average Precision (MAP): 0.3439
